### Prediction with trained ANN model

In [82]:
import numpy as np
import pandas as pd
import pickle

import tensorflow as tf
from tensorflow.keras.models import load_model

In [83]:
# Load preprocessors and trained model
model = load_model('./dl_model/classification_model.h5')

with open('./preprocessors/classification/gender_label_encoder.pkl', 'rb') as file:
    gender_encoder = pickle.load(file)

with open('./preprocessors/classification/onehot_geo_encoder.pkl', 'rb') as file:
    geo_encoder = pickle.load(file)

with open('./preprocessors/classification/standard_scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [88]:
geo_encoder.categories_[0], gender_encoder.classes_

(array(['France', 'Germany', 'Spain'], dtype=object),
 array(['Female', 'Male'], dtype=object))

In [70]:
# Example input data
input_data = {'CreditScore': 600,
              'Geography': 'France',
              'Gender': 'Male',
              'Age': 40,
              'Tenure': 3,
              'Balance': 60000,
              'NumOfProducts': 2,
              'HasCrCard': 1,
              'IsActiveMember': 1,
              'EstimatedSalary': 50000}

In [71]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [91]:
gender_encoder.transform(['Male'])[0]

np.int64(1)

In [99]:
geo_encoder.transform([['France']])[0]

/home/htet-aung-lynn/Study/E2E-Churn-Prediction-with-ANN/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


array([1., 0., 0.])

In [72]:
geo_encoded_df = pd.DataFrame(data = geo_encoder.transform(input_df[['Geography']]), 
                              columns = geo_encoder.get_feature_names_out())

input_df = pd.concat([input_df, geo_encoded_df], axis=1)

input_df['Gender'] = gender_encoder.transform(input_df['Gender'])
input_df.drop(['Geography'], axis=1, inplace=True)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [74]:
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [76]:
# Predict churn
prediction = model.predict(input_scaled)
prediction_proba = prediction[0][0]
prediction_proba

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step


np.float32(0.03954284)

In [77]:
if prediction_proba > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")

The customer is not likely to churn.
